# Import

In [1]:
!pip install llmcompressor
!pip install -U "transformers==4.57.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 282.1/282.1 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 136.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


In [2]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor.modifiers.smoothquant import SmoothQuantModifier
from compressed_tensors.quantization.quant_args import ActivationOrdering

# Setting

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**LGAI-EXAONE/MANTA-1M**

- 출처: Hugging Face Datasets

- 라이선스: CC-BY-NC-4.0 (비상업적 사용 허용)

- 행(row) 수: 1,000,000개

- 파일 크기: 약 1.94 GB (Parquet 포함)
- 다양한 분야의 지식과 관련된 대화 형식의 corpus

컬럼 구조:

- id: 각 샘플의 고유 ID

- conversations: 구조화된 instruction → response 형태의 쌍 (대화/질문 + 답변)

- complexity_label: 난이도/복잡도 지표

In [4]:
MODEL_ID = "/content/drive/MyDrive/base_model"
OUT_DIR  = "/content/drive/MyDrive/GPTQ_models/smooth_35_1024"

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 1024
MAX_SEQUENCE_LENGTH = 512

SCHEME = "W8A8"
TARGETS = ["Linear"]

IGNORE = [
    "embed_tokens",
    "lm_head",
    "norm",                  # model.norm
    "input_layernorm",       # per-layer
    "post_attention_layernorm"
]

# Model Loads

In [5]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    trust_remote_code=True,
)

print("[INFO] 모델/토크나이저 로드 완료")

[INFO] 모델 로드 중...
[INFO] 모델/토크나이저 로드 완료


# Dataset Loads & Preprocess

- gsm8k calibration용
- dataset -> input_id로 변경하는 pipeline

- to_message 함수: exaone과 같은 모델은 tokenizer에 chat template이 존재
- 해당 템플릿에 데이터셋을 맞추기 위한 함수
- tokenizer.apply_chat_template(): 위 함수를 거친 messages list를 받아 모델이 기대하는(tokenizer에 있는) prompt 형식으로 제작

- 대화형 dataset용

In [6]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
   return {
       "text": tokenizer.apply_chat_template(
           example["conversations"],
           add_generation_prompt=True,
           tokenize=False)
          }



ds = ds.map(preprocess)

print("[INFO] 데이터 전처리 완료")

[INFO] 캘리브레이션 데이터 로드 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train.parquet:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1024 [00:00<?, ? examples/s]

[INFO] 데이터 전처리 완료


# GPTQ Quantization

In [7]:
print(f"[INFO] GPTQ 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

IGNORE = [
    "embed_tokens",
    "lm_head",
    "norm",                  # model.norm
    "input_layernorm",       # per-layer
    "post_attention_layernorm"
]


recipe = [
    SmoothQuantModifier(
        smoothing_strength=0.35,
        mappings=[
            [["mlp.gate_proj", "mlp.up_proj"], "post_attention_layernorm"],
        ]
    ),
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] GPTQ 완료")

[INFO] GPTQ 시작 (scheme=W8A8, samples=1024, max_len=512)...


Tokenizing:   0%|          | 0/1024 [00:00<?, ? examples/s]

2026-02-24T16:34:17.741315+0000 | reset | INFO - Compression lifecycle reset
2026-02-24T16:34:17.744252+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-24T16:34:17.786957+0000 | initialize | INFO - Compression lifecycle initialized for 2 modifiers
2026-02-24T16:34:17.788094+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `SmoothQuantModifier`


(31/31): Propagating: 100%|██████████| 1024/1024 [00:00<00:00, 1029.55it/s]

2026-02-24T16:37:40.651379+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`


...

(30/31): Calibrating: 100%|██████████| 1024/1024 [00:05<00:00, 199.68it/s]

2026-02-24T16:46:03.894977+0000 | compress_modules | INFO - Quantizing model.layers.29.self_attn.q_proj using 1024 samples


2026-02-24T16:46:04.963683+0000 | compress | METRIC - time 1.06s
2026-02-24T16:46:04.966054+0000 | compress | METRIC - error 38.76
2026-02-24T16:46:04.967282+0000 | compress | METRIC - GPU 0 | usage: 2.06% | total memory: 85 GB
2026-02-24T16:46:04.968010+0000 | compress | METRIC - Compressed module size: 8.394752 MB
2026-02-24T16:46:04.969070+0000 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 1024 samples
2026-02-24T16:46:06.022687+0000 | compress | METRIC - time 1.05s
2026-02-24T16:46:06.024685+0000 | compress | METRIC - error 14.19
2026-02-24T16:46:06.026157+0000 | compress | METRIC - GPU 0 | usage: 2.06% | total memory: 85 GB
2026-02-24T16:46:06.026789+0000 | compress | METRIC - Compressed module size: 2.098688 MB
2026-02-24T16:46:06.027949+0000 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 1024 samples
2026-02-24T16:46:07.088068+0000 | compress | METRIC - time 1.06s
2026-02-24T16:46:07.090349+0000 | compress | METRIC -

(31/31): Propagating: 100%|██████████| 1024/1024 [00:01<00:00, 1021.77it/s]


2026-02-24T16:46:17.943879+0000 | finalize | INFO - Compression lifecycle finalized for 2 modifiers
2026-02-24T16:46:18.002455+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
[INFO] GPTQ 완료


# Model Save

In [8]:
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(
                      OUT_DIR,
                      safe_serialization=True
                      )
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")

2026-02-24T16:46:18.059968+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:01, 158.40it/s]


[INFO] 모델 저장 완료: /content/drive/MyDrive/GPTQ_models/smooth_35_1024


- Save 후, config 확인 필요

In [ ]:
import inspect
print(inspect.signature(GPTQModifier))
#actorder=ActivationOrdering.DYNAMIC,